# GOAL IS TO FIND Amount of T16 Parts in Adelshofen

* Translation: Amount of ID_T16 in Gemeinde == Adelshofen

## 3.1 Important tables
* [Bestandteile_Komponente_K2ST2.csv](data\Komponente\Bestandteile_Komponente_K2ST2.csv) contains "ID_T16" and "ID_K2ST2"
* [Bestandteile_Fahrzeuge_OEM1_Typ11.csv](data\Fahrzeug\Bestandteile_Fahrzeuge_OEM1_Typ11.csv) contains "ID_Sitze" and "ID_Fahrzeug"
* [Zulassungen_alle_Fahrzeuge.csv](data\Zulassungen\Zulassungen_alle_Fahrzeuge.csv) contains "IDNummer" and "Gemeinden"


## 3.2 Pipeline
* "ID_T16" -> "ID_K2ST2" == "ID_Sitze"
* "ID_Sitze" -> "ID_Fahrzeug" == "IDNummer"
* "IDNummer" -> "Gemeinden"

## 3.3 How to merge
* -> means both columns are in same table refer to 3.1
* = means they are the same column. They have different column names. So we use the arguments left_on="", right_on="" in the pandas merge
* only merge necessary columns

## 3.4 Example
* merged table could look like this:
* ID_T16, ID_Sitze, ID_Fahrzeug, Gemeinde
* Then we should be able to Count distinct how many Parts in Adelshofen we have
* Important INFOS: 
* [Bestandteile_Fahrzeuge_OEM1_Typ11.csv](data\Fahrzeug\Bestandteile_Fahrzeuge_OEM1_Typ11.csv) is not the only table. We have to look at [Bestandteile_Fahrzeuge_OEM1_Typ12.csv](data\Fahrzeug\Bestandteile_Fahrzeuge_OEM1_Typ12.csv), [Bestandteile_Fahrzeuge_OEM2_Typ21.csv](data\Fahrzeug\Bestandteile_Fahrzeuge_OEM2_Typ21.csv), [Bestandteile_Fahrzeuge_OEM2_Typ22.csv](data\Fahrzeug\Bestandteile_Fahrzeuge_OEM2_Typ22.csv). Because the same table is split in 4
* [data\Komponente\Bestandteile_Komponente_K2LE2.csv](data\Komponente\Bestandteile_Komponente_K2LE2.csv) contains "ID_T16" aswell so we have to check through all Bestandteile files for columns with T16
* its important to combines these first before merging so we are not missing any Parts
* T16 table is a txt file and poorly formated so it has to be dealt with correctly

In [3]:
import pandas as pd

# -------------------------------------------------
# Registration table
# -------------------------------------------------
registration = pd.read_csv(
    "data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv",
    sep=";",
    engine="python",
    usecols=["IDNummer", "Gemeinden"]
)

registration.columns = registration.columns.str.strip()

# -------------------------------------------------
# Read T16 -> Seat mapping
# -------------------------------------------------
k2st2 = pd.read_csv(
    "data/Komponente/Bestandteile_Komponente_K2ST2.csv",
    sep=";",
    engine="python",
    usecols=["ID_T16", "ID_K2ST2"]
)

# -------------------------------------------------
# Vehicle-Part files
# -------------------------------------------------
vehicle_files = [
    "data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv",
    "data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv",
    "data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ21.csv",
    "data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ22.csv"
]

result_chunks = []

for file in vehicle_files:

    print("Reading:", file)

    for chunk in pd.read_csv(
        file,
        sep=";",
        engine="python",
        chunksize=50000,
        usecols=["ID_Sitze", "ID_Fahrzeug"]
    ):

        merged = chunk.merge(
            k2st2,
            left_on="ID_Sitze",
            right_on="ID_K2ST2",
            how="inner"
        )

        merged = merged.merge(
            registration,
            left_on="ID_Fahrzeug",
            right_on="IDNummer",
            how="inner"
        )

        result_chunks.append(
            merged[["ID_T16", "Gemeinden"]]
        )

# -------------------------------------------------
# Combine results
# -------------------------------------------------
result = pd.concat(result_chunks, ignore_index=True)

# -------------------------------------------------
# Filter Adelshofen
# -------------------------------------------------
adelshofen = result[
    result["Gemeinden"].str.upper() == "ADELSHOFEN"
]

# -------------------------------------------------
# Count unique T16
# -------------------------------------------------
count = adelshofen["ID_T16"].nunique()

print()
print("Number of unique T16 parts in Adelshofen:", count)

Reading: data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv
Reading: data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv
Reading: data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ21.csv
Reading: data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ22.csv

Number of unique T16 parts in Adelshofen: 36
